# Step 1: Import Data

In [12]:
import pandas as pd
import re

In [13]:
file_path = r'C:\Users\cau.tran\OneDrive\2. Study\13. Data Engineering with Unigap\Action 1\data.csv'

In [14]:
# file_path = r'C:\Users\Windows\OneDrive\2. Study\13. Data Engineering with Unigap\Action 1\data.csv'

In [15]:
raw = pd.read_csv(file_path)

In [16]:
raw.head()  # Display the first few rows of the DataFrame

,created_date,job_title,company,salary,address,time,link_description
0,2023-08-01,Business Analyst,Công ty TNHH Công nghệ số Adamo,10 - 20 triệu,Hà Nội,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/business-analyst...
1,2023-08-01,Nhân Viên Lập Trình Phần Mềm - Thu Nhập Từ 10 ...,Công ty TNHH Đầu Tư Công Nghệ ST,10 - 20 triệu,Hồ Chí Minh,Còn 91 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/nhan-vien-lap-tr...
2,2023-08-01,.Net Developer (N3) | T9160,Công ty TNHH CMC GLOBAL,Thoả thuận,Toàn Quốc,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/net-developer-n3...
3,2023-08-01,"Project Manager (Tiếng Anh Giao Tiếp, Từ 1 Năm...",CÔNG TY CỔ PHẦN CÔNG NGHỆ SOTATEK,Tới 35 triệu,Hà Nội: Cầu Giấy,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/project-manager-...
4,2023-08-01,Devops/Sre - Chuyên Viên Quản Trị Hệ Thống/Aut...,Công ty Cổ phần Thời Trang Yody,Tới 50 triệu,Hà Nội: Thanh Xuân: Hải Dương: TP Hải Dương,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/devops-sre-chuye...


In [17]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1933 entries, 0 to 1932
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   created_date      1933 non-null   object
 1   job_title         1933 non-null   object
 2   company           1933 non-null   object
 3   salary            1933 non-null   object
 4   address           1933 non-null   object
 5   time              1933 non-null   object
 6   link_description  1933 non-null   object
dtypes: object(7)
memory usage: 105.8+ KB


# Step 2: Data cleaning

## Yêu cầu 1: Chuẩn hóa và làm sạch dữ liệu

- Chuẩn hóa cột salary về dạng số, xử lý các giá trị như "Thoả thuận", "Trên X triệu", "X - Y triệu", "Tới X triệu", etc.
- Tạo thêm các cột phụ: min_salary, max_salary, salary_unit (VND/USD)
- Xử lý cột address để tách thành city và district
- Chuẩn hóa job_title để gom nhóm các vị trí tương tự (ví dụ: "Software Engineer", "Developer", "Programmer" có thể gom vào một nhóm)

Question

- Lương thỏa thuận thống nhất rule tách ra như thế nào? (để null hay giá trị 0) 
- Cột salary chứa text thì không format được sang dạng số >> Đề xuất khi tách ra 2 cột min & max sẽ format về dạng số
- Best practises để đặt tên các biến khi code?
- Tại sao lại có nhiều tỉnh, quận cho 1 dòng?
- Các để chuẩn hóa job_title nhanh

In [18]:
raw['salary'].unique()  # Check unique values in the salary column

array(['10 - 20 triệu', 'Thoả thuận', 'Tới 35 triệu', 'Tới 50 triệu',
       '12 - 18 triệu', '2 - 6 triệu', '3 - 6 triệu', 'Trên 1,000 USD',
       '10 - 30 triệu', '9 - 10 triệu', '30 - 35 triệu', '8 - 12 triệu',
       '9 - 12 triệu', '10 - 18 triệu', '600 - 1,300 USD', '3 - 8 triệu',
       'Trên 3 triệu', 'Tới 2,000 USD', 'Tới 40 triệu', '20 - 30 triệu',
       'Trên 12 triệu', 'Tới 30 triệu', '18 - 25 triệu', '7 - 8 triệu',
       'Tới 1,000 USD', '7 - 10 triệu', '20 - 40 triệu', '8 - 15 triệu',
       'Tới 750 USD', 'Tới 28 triệu', '12 - 20 triệu', '13 - 18 triệu',
       '1,000 - 2,000 USD', '13 - 20 triệu', 'Tới 1,500 USD',
       'Trên 20 triệu', '25 - 40 triệu', 'Tới 3 triệu', 'Tới 2 triệu',
       'Tới 20 triệu', 'Tới 3,000 USD', 'Tới 1,300 USD', '17 - 22 triệu',
       'Tới 15 triệu', '7 - 15 triệu', '3 - 5 triệu', '700 - 1,500 USD',
       'Tới 25 triệu', '10 - 15 triệu', 'Trên 10 triệu', 'Trên 1,700 USD',
       '20 - 35 triệu', '13 - 22 triệu', '10 - 12 triệu', 'Tới 1,8

In [19]:
df1 = raw.copy()

In [20]:
# Hàm xử lý salary
def extract_salary_range(s):
    s = s.lower().replace(',', '').strip()

    if 'usd' in s:
        currency = 'USD'
    elif 'triệu' in s:
        currency = 'VND'
    else:
        currency = 'Other'
    
    if 'thoả thuận' in s or 'thỏa thuận' in s:
        return None, None, currency

    if 'trên' in s:
        num = re.findall(r"[\d.]+", s)
        if num:
            val = float(num[0])
            return round(val, 2), None, currency

    if 'tới' in s or 'to' in s:
        num = re.findall(r"[\d.]+", s)
        if num:
            val = float(num[0])
            return None, round(val, 2), currency

    match = re.findall(r"[\d.]+", s)
    if len(match) == 2:
        return round(float(match[0]), 2), round(float(match[1]), 2), currency

    return None, None, currency

# Áp dụng tách lương
df1[['min_salary', 'max_salary','unit_currency']] = df1['salary'].apply(lambda x: pd.Series(extract_salary_range(x)))

# Kết quả
df1[['salary','min_salary','max_salary','unit_currency']] 

,salary,min_salary,max_salary,unit_currency
0,10 - 20 triệu,10.0,20.0,VND
1,10 - 20 triệu,10.0,20.0,VND
2,Thoả thuận,NaN,NaN,Other
3,Tới 35 triệu,NaN,35.0,VND
4,Tới 50 triệu,NaN,50.0,VND
...,...,...,...,...
1928,Thoả thuận,NaN,NaN,Other
1929,Trên 3 triệu,3.0,NaN,VND
1930,Tới 50 triệu,NaN,50.0,VND
1931,Thoả thuận,NaN,NaN,Other


In [21]:
df1.info()  # Check the DataFrame info to see the new columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1933 entries, 0 to 1932
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   created_date      1933 non-null   object 
 1   job_title         1933 non-null   object 
 2   company           1933 non-null   object 
 3   salary            1933 non-null   object 
 4   address           1933 non-null   object 
 5   time              1933 non-null   object 
 6   link_description  1933 non-null   object 
 7   min_salary        709 non-null    float64
 8   max_salary        1098 non-null   float64
 9   unit_currency     1933 non-null   object 
dtypes: float64(2), object(8)
memory usage: 151.1+ KB


In [22]:
def split_address_pairs(address):
    parts = [part.strip() for part in address.split(':')]
    result = {}
    for i in range(0, len(parts), 2):
        if i + 1 < len(parts):
            city_col = f'city_{i//2 + 1}'
            dist_col = f'district_{i//2 + 1}'
            result[city_col] = parts[i]
            result[dist_col] = parts[i + 1]
        else:
            # Nếu lẻ thì phần cuối cùng không có cặp -> gán riêng nếu muốn
            result[f'city_{i//2 + 1}'] = parts[i]
    return pd.Series(result)

# Áp dụng
df_split = df1['address'].apply(split_address_pairs)

# Gộp kết quả lại
df2 = df1.join(df_split)

df2.head()  # Hiển thị kết quả

,created_date,job_title,company,salary,address,time,link_description,min_salary,max_salary,unit_currency,city_1,district_1,city_2,district_2,city_3,district_3,city_4,district_4
0,2023-08-01,Business Analyst,Công ty TNHH Công nghệ số Adamo,10 - 20 triệu,Hà Nội,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/business-analyst...,10.0,20.0,VND,Hà Nội,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-08-01,Nhân Viên Lập Trình Phần Mềm - Thu Nhập Từ 10 ...,Công ty TNHH Đầu Tư Công Nghệ ST,10 - 20 triệu,Hồ Chí Minh,Còn 91 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/nhan-vien-lap-tr...,10.0,20.0,VND,Hồ Chí Minh,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-08-01,.Net Developer (N3) | T9160,Công ty TNHH CMC GLOBAL,Thoả thuận,Toàn Quốc,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/net-developer-n3...,NaN,NaN,Other,Toàn Quốc,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-08-01,"Project Manager (Tiếng Anh Giao Tiếp, Từ 1 Năm...",CÔNG TY CỔ PHẦN CÔNG NGHỆ SOTATEK,Tới 35 triệu,Hà Nội: Cầu Giấy,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/project-manager-...,NaN,35.0,VND,Hà Nội,Cầu Giấy,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-08-01,Devops/Sre - Chuyên Viên Quản Trị Hệ Thống/Aut...,Công ty Cổ phần Thời Trang Yody,Tới 50 triệu,Hà Nội: Thanh Xuân: Hải Dương: TP Hải Dương,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/devops-sre-chuye...,NaN,50.0,VND,Hà Nội,Thanh Xuân,Hải Dương,TP Hải Dương,NaN,NaN,NaN,NaN


In [23]:
# Kiểm tra lại data tỉnh, quận
# df2.to_csv(r'C:\Users\Windows\OneDrive\2. Study\13. Data Engineering with Unigap\Action 1\data_processed.csv', index=False, encoding='utf-8-sig') 

In [24]:
# Kiểm tra các giá trị duy nhất trong cột 'job_title'
# df2['job_title'].to_csv(r'C:\Users\Windows\OneDrive\2. Study\13. Data Engineering with Unigap\Action 1\job_title.csv', index=False, encoding='utf-8-sig')  # Lưu job_title vào file CSV

In [25]:
group_keywords = {
    'Software Engineer': ['developer', 'software engineer', 'programmer', 'coder', 'lập trình'],
    'Data': ['data', 'analyst', 'bi', 'business analyst'],
    'DevOps': ['devops', 'sre', 'system admin'],
    'PM': ['project manager', 'scrum master', 'product owner'],
    'IT Support': ['it support', 'helpdesk', 'technical support', 'application support'],
    'Tester': ['qa', 'qc', 'tester', 'kiểm thử'],
    'Intern': ['thực tập', 'intern'],
    'Other': []  # fallback nếu không khớp gì
}

nlp - natural language processing

fuzzy - khớp bao nhiêu % (tách từng ký tự)



In [26]:
def normalize_job_title(title):
    title = str(title).lower()
    for group, keywords in group_keywords.items():
        if any(keyword in title for keyword in keywords):
            return group
    return 'Other'

In [27]:
df2['job_group'] = df2['job_title'].apply(normalize_job_title)

In [28]:
df2.head()  # Hiển thị kết quả

,created_date,job_title,company,salary,address,time,link_description,min_salary,max_salary,unit_currency,city_1,district_1,city_2,district_2,city_3,district_3,city_4,district_4,job_group
0,2023-08-01,Business Analyst,Công ty TNHH Công nghệ số Adamo,10 - 20 triệu,Hà Nội,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/business-analyst...,10.0,20.0,VND,Hà Nội,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data
1,2023-08-01,Nhân Viên Lập Trình Phần Mềm - Thu Nhập Từ 10 ...,Công ty TNHH Đầu Tư Công Nghệ ST,10 - 20 triệu,Hồ Chí Minh,Còn 91 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/nhan-vien-lap-tr...,10.0,20.0,VND,Hồ Chí Minh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Software Engineer
2,2023-08-01,.Net Developer (N3) | T9160,Công ty TNHH CMC GLOBAL,Thoả thuận,Toàn Quốc,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/net-developer-n3...,NaN,NaN,Other,Toàn Quốc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Software Engineer
3,2023-08-01,"Project Manager (Tiếng Anh Giao Tiếp, Từ 1 Năm...",CÔNG TY CỔ PHẦN CÔNG NGHỆ SOTATEK,Tới 35 triệu,Hà Nội: Cầu Giấy,Còn 25 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/project-manager-...,NaN,35.0,VND,Hà Nội,Cầu Giấy,NaN,NaN,NaN,NaN,NaN,NaN,PM
4,2023-08-01,Devops/Sre - Chuyên Viên Quản Trị Hệ Thống/Aut...,Công ty Cổ phần Thời Trang Yody,Tới 50 triệu,Hà Nội: Thanh Xuân: Hải Dương: TP Hải Dương,Còn 30 ngày để ứng tuyển,https://www.topcv.vn/viec-lam/devops-sre-chuye...,NaN,50.0,VND,Hà Nội,Thanh Xuân,Hải Dương,TP Hải Dương,NaN,NaN,NaN,NaN,DevOps
